In [ ]:
import os, json, time, random
from dataclasses import dataclass
from typing import List, Dict, Tuple

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from tqdm.auto import tqdm

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

@dataclass
class CFG:
    model_id: str = "sshleifer/tiny-gpt2"  # VERY small
    out_dir: str = "./tiny_socratic_rlhf"

    # short sequences
    max_prompt_len: int = 192
    max_new_tokens: int = 48

    # preference collection
    num_pref: int = 16
    temp_A: float = 0.7
    temp_B: float = 1.1
    top_p: float = 0.95

    # reward model
    rm_epochs: int = 10
    rm_lr: float = 3e-3
    rm_batch: int = 4
    rm_max_seq_len: int = 256
    freeze_rm_backbone: bool = True

    # RLHF
    rlhf_steps: int = 80
    rlhf_lr: float = 2e-4
    beta_kl: float = 0.05
    rollout_batch: int = 2
    logprob_microbatch: int = 1   # max memory safety
    seed: int = 42

cfg = CFG()
set_seed(cfg.seed)

os.makedirs(cfg.out_dir, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
TASK = """You are a Socratic micro-tutor.
Given a student's misconception, ask exactly ONE Socratic question (end with '?').
Constraints:
- <= 18 words
- do not directly give the answer
Optional short hint in parentheses after the question.
"""

PROMPTS = [
    ("Probability", "If events are independent then P(A|B)=P(B|A)."),
    ("Linear Algebra", "If Ax=b has a solution, it must be unique."),
    ("Calculus", "If f'(x)=0, the point is always a maximum."),
    ("ML", "More epochs always improve test accuracy."),
    ("Optimization", "Gradient descent always finds global minimum."),
    ("Stats", "95% CI means 95% chance parameter is inside the interval."),
    ("NLP", "Attention weights always explain model decisions."),
    ("Neural Nets", "Deeper networks always overfit more."),
    ("Graphs", "High-degree nodes must be close in shortest path."),
    ("Info Theory", "Entropy is maximized when probabilities are unequal."),
    ("Causality", "Correlation implies causation if p-value is small."),
    ("Geometry", "All norms give same distances in R^n."),
]

def build_prompt(topic, misconception):
    return (
        TASK.strip()
        + "\n\n"
        + f"Topic: {topic}\n"
        + f"Student: {misconception}\n"
        + "Your question:"
    )

prompt_texts = [build_prompt(t, m) for (t, m) in PROMPTS]
print(prompt_texts[0])

You are a Socratic micro-tutor.
Given a student's misconception, ask exactly ONE Socratic question (end with '?').
Constraints:
- <= 18 words
- do not directly give the answer
Optional short hint in parentheses after the question.

Topic: Probability
Student: If events are independent then P(A|B)=P(B|A).
Your question:


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(cfg.model_id, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

policy = AutoModelForCausalLM.from_pretrained(cfg.model_id, torch_dtype=torch.float32).to(device)
policy.config.use_cache = False
policy.train()

pi_ref = AutoModelForCausalLM.from_pretrained(cfg.model_id, torch_dtype=torch.float32).to(device)
pi_ref.config.use_cache = False
pi_ref.eval()
for p in pi_ref.parameters():
    p.requires_grad = False

print("Loaded:", cfg.model_id)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/2.51M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.51M [00:00<?, ?B/s]

Loaded: sshleifer/tiny-gpt2


In [ ]:
@torch.no_grad()
def generate_one(model, prompt: str, temp: float) -> str:
    tokenizer.padding_side = "left"
    enc = tokenizer([prompt], return_tensors="pt", padding=True,
                    truncation=True, max_length=cfg.max_prompt_len).to(device)
    inlen = enc["input_ids"].shape[1]
    out = model.generate(
        **enc,
        do_sample=True,
        temperature=temp,
        top_p=cfg.top_p,
        max_new_tokens=cfg.max_new_tokens,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    gen = out[:, inlen:]
    txt = tokenizer.batch_decode(gen, skip_special_tokens=True)[0].strip()
    txt = txt.split("\n")[0].strip()
    if "?" in txt:
        txt = txt[:txt.index("?")+1].strip()
    else:
        txt = (txt.rstrip(".") + "?").strip()
    w = txt.split()
    if len(w) > 18:
        txt = " ".join(w[:18])
        if not txt.endswith("?"):
            txt = txt.rstrip(".") + "?"
    return txt

def generate_pair(prompt: str):
    a = generate_one(policy, prompt, cfg.temp_A)
    b = generate_one(policy, prompt, cfg.temp_B)
    if a == b:
        b = generate_one(policy, prompt, cfg.temp_B + 0.2)
    return a, b

def encode_texts(texts: List[str], max_len: int):
    tokenizer.padding_side = "right"
    enc = tokenizer(texts, return_tensors="pt", padding=True,
                    truncation=True, max_length=max_len)
    return {k: v.to(device) for k, v in enc.items()}

def gen_valid_mask(gen_ids: torch.Tensor):
    pad_id = tokenizer.pad_token_id
    eos_id = tokenizer.eos_token_id
    not_pad = (gen_ids != pad_id)
    eos = (gen_ids == eos_id).long()
    eos_cum = torch.cumsum(eos, dim=1)
    upto = (eos_cum <= 1)
    return (not_pad & upto).float()

def sum_logprob_generated_safe(model, full_ids: torch.Tensor, input_len: int):
    """
    log p(token) = logit[token] - logsumexp(logits)
    avoids building log_softmax tensor.
    Returns:
      logp_sum [B], T [B]
    """
    pad_id = tokenizer.pad_token_id
    attn = (full_ids != pad_id).long()
    out = model(input_ids=full_ids, attention_mask=attn, return_dict=True)
    logits = out.logits  # [B,T,V]

    next_tok = full_ids[:, 1:]          # [B,T-1]
    logits_next = logits[:, :-1, :]     # [B,T-1,V]

    tok_logit = logits_next.gather(-1, next_tok.unsqueeze(-1)).squeeze(-1)  # [B,T-1]
    lse = torch.logsumexp(logits_next, dim=-1)                               # [B,T-1]
    tok_logp = tok_logit - lse                                               # [B,T-1]

    gen_ids = full_ids[:, input_len:]  # [B,G]
    B, G = gen_ids.shape
    if G == 0:
        return torch.zeros(B, device=device), torch.ones(B, device=device)

    valid = gen_valid_mask(gen_ids)    # [B,G]
    T = valid.sum(dim=1).clamp(min=1.0)

    start = max(input_len - 1, 0)
    end = start + G
    tok_logp_gen = tok_logp[:, start:end]
    if tok_logp_gen.shape[1] != G:
        G2 = min(G, tok_logp_gen.shape[1])
        tok_logp_gen = tok_logp_gen[:, :G2]
        valid = valid[:, :G2]
        T = valid.sum(dim=1).clamp(min=1.0)

    return (tok_logp_gen * valid).sum(dim=1), T

def clear_cuda():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
pref_path = os.path.join(cfg.out_dir, "prefs.jsonl")

def collect_prefs(n: int):
    print("Label preferences: 1=A, 2=B, 0=skip\n")
    idxs = np.random.choice(len(prompt_texts), size=min(n, len(prompt_texts)), replace=False)
    saved = 0
    with open(pref_path, "a", encoding="utf-8") as f:
        for k, idx in enumerate(idxs, start=1):
            prompt = prompt_texts[int(idx)]
            a, b = generate_pair(prompt)

            print("="*70)
            print(f"[{k}/{len(idxs)}] Topic:", PROMPTS[int(idx)][0])
            print("Misconception:", PROMPTS[int(idx)][1])
            print("-"*70)
            print("A:", a)
            print("B:", b)

            c = None
            while c not in ("0","1","2"):
                c = input("Choice (1/2/0): ").strip()

            rec = {"prompt_id": int(idx), "prompt": prompt, "A": a, "B": b, "choice": int(c)}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            f.flush()

            if c in ("1","2"):
                saved += 1

    print("\nSaved usable prefs:", saved, "| file:", pref_path)

collect_prefs(cfg.num_pref)

Label preferences: 1=A, 2=B, 0=skip

[1/12] Topic: Causality
Misconception: Correlation implies causation if p-value is small.
----------------------------------------------------------------------
A: autonomy Observ antibioticoho circumcised heirimura confir ESV TA Prob trilogy DanielRocket credibilitydit Money Habit Money conservation trilogy scalp?
B: TAtingoother004 Daniel reviewing heir vendorsmediately confir hauledoho antibioticatisf Rhreement credibility subst circumcised confirditJD ESV heir trilogy vendors haulediken?
Choice (1/2/0): 1
[2/12] Topic: Info Theory
Misconception: Entropy is maximized when probabilities are unequal.
----------------------------------------------------------------------
A: Motorola Money heirimuraatisf TA Money hauledreementimura hauledSher pawnRocket Brew conservation pawnhibit ObservSceneScenepress circumcisedimura pawn heir Daniel vendors Motoroladit?
B: hauled Prob vendors conservation heirRocket Hancock TAmediately confir conservationtingoothe

In [ ]:
def load_jsonl(path):
    rows = []
    if not os.path.isfile(path):
        return rows
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

rows = load_jsonl(pref_path)
pairs = []
for r in rows:
    if r["choice"] == 1:
        chosen, rejected = r["A"], r["B"]
    elif r["choice"] == 2:
        chosen, rejected = r["B"], r["A"]
    else:
        continue
    pairs.append({
        "prompt": r["prompt"],
        "chosen_text": r["prompt"] + " " + chosen,
        "rejected_text": r["prompt"] + " " + rejected,
    })

print("Usable pairs:", len(pairs))
if len(pairs) < 6:
    print("Too few pairs—collect more in Cell 6 for meaningful RM training.")

np.random.shuffle(pairs)
split = max(1, int(0.8 * len(pairs)))
train_pairs = pairs[:split]
eval_pairs = pairs[split:] if len(pairs) - split >= 2 else pairs[:2]
print("Train:", len(train_pairs), "Eval:", len(eval_pairs))

Usable pairs: 11
Train: 8 Eval: 3


In [ ]:
class RewardModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = AutoModelForCausalLM.from_pretrained(cfg.model_id, torch_dtype=torch.float32).to(device)
        self.backbone.config.use_cache = False

        hs = getattr(self.backbone.config, "hidden_size", None)
        if hs is None:
            hs = getattr(self.backbone.config, "n_embd", None)
        self.head = nn.Linear(hs, 1).to(device)

        if cfg.freeze_rm_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask,
                            output_hidden_states=True, return_dict=True)
        H = out.hidden_states[-1]
        last = attention_mask.long().sum(dim=1) - 1
        last = torch.clamp(last, min=0)
        bidx = torch.arange(H.size(0), device=H.device)
        h_last = H[bidx, last, :]
        return self.head(h_last).squeeze(-1)

rm = RewardModel()
opt = torch.optim.AdamW([p for p in rm.parameters() if p.requires_grad], lr=cfg.rm_lr)

def bt_loss(r_c, r_r):
    return F.softplus(-(r_c - r_r)).mean()

def batched(lst, bs):
    for i in range(0, len(lst), bs):
        yield lst[i:i+bs]

@torch.no_grad()
def eval_rm(pairs_list):
    rm.eval()
    L, A = [], []
    for b in batched(pairs_list, cfg.rm_batch):
        c = encode_texts([x["chosen_text"] for x in b], cfg.rm_max_seq_len)
        r = encode_texts([x["rejected_text"] for x in b], cfg.rm_max_seq_len)
        rc = rm(c["input_ids"], c["attention_mask"])
        rr = rm(r["input_ids"], r["attention_mask"])
        L.append(bt_loss(rc, rr).item())
        A.append((rc > rr).float().mean().item())
    rm.train()
    return float(np.mean(L)), float(np.mean(A))

print("RM eval before:", eval_rm(eval_pairs))

rm.train()
for ep in range(cfg.rm_epochs):
    random.shuffle(train_pairs)
    for b in batched(train_pairs, cfg.rm_batch):
        c = encode_texts([x["chosen_text"] for x in b], cfg.rm_max_seq_len)
        r = encode_texts([x["rejected_text"] for x in b], cfg.rm_max_seq_len)
        rc = rm(c["input_ids"], c["attention_mask"])
        rr = rm(r["input_ids"], r["attention_mask"])
        loss = bt_loss(rc, rr)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    if (ep+1) % 2 == 0:
        print(f"Epoch {ep+1}/{cfg.rm_epochs} eval:", eval_rm(eval_pairs))

print("RM eval after:", eval_rm(eval_pairs))

rm_dir = os.path.join(cfg.out_dir, "rm")
os.makedirs(rm_dir, exist_ok=True)
torch.save(rm.head.state_dict(), os.path.join(rm_dir, "head.pt"))
print("Saved RM head:", rm_dir)

RM eval before: (0.6902574896812439, 1.0)
Epoch 2/10 eval: (0.6916345357894897, 1.0)
Epoch 4/10 eval: (0.692906379699707, 1.0)
Epoch 6/10 eval: (0.694030225276947, 0.0)
Epoch 8/10 eval: (0.6949748992919922, 0.0)
Epoch 10/10 eval: (0.6961666345596313, 0.0)
RM eval after: (0.6961666345596313, 0.0)
Saved RM head: ./tiny_socratic_rlhf/rm


In [ ]:
policy.train()
opt_pi = torch.optim.AdamW(policy.parameters(), lr=cfg.rlhf_lr)

@torch.no_grad()
def rm_score(texts: List[str]) -> torch.Tensor:
    rm.eval()
    enc = encode_texts(texts, cfg.rm_max_seq_len)
    s = rm(enc["input_ids"], enc["attention_mask"]).float()
    rm.train()
    return s

def rollout(prompts: List[str], temp: float):
    tokenizer.padding_side = "left"
    enc = tokenizer(prompts, return_tensors="pt", padding=True,
                    truncation=True, max_length=cfg.max_prompt_len).to(device)
    inlen = enc["input_ids"].shape[1]
    with torch.no_grad():
        full = policy.generate(
            **enc, do_sample=True, temperature=temp, top_p=cfg.top_p,
            max_new_tokens=cfg.max_new_tokens,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id
        )
    gen = full[:, inlen:]
    texts = tokenizer.batch_decode(gen, skip_special_tokens=True)
    texts = [t.strip().split("\n")[0].strip() for t in texts]
    cleaned = []
    for t in texts:
        if "?" in t:
            t = t[:t.index("?")+1].strip()
        else:
            t = (t.rstrip(".") + "?").strip()
        w = t.split()
        if len(w) > 18:
            t = " ".join(w[:18])
            if not t.endswith("?"):
                t = t.rstrip(".") + "?"
        cleaned.append(t)
    return full, inlen, cleaned

logs = []
pbar = tqdm(range(cfg.rlhf_steps), desc="Tiny RLHF")
for step in pbar:
    idxs = np.random.choice(len(prompt_texts), size=cfg.rollout_batch, replace=True)
    ps = [prompt_texts[int(i)] for i in idxs]

    full, inlen, outs = rollout(ps, temp=0.9)
    texts = [p + " " + o for p, o in zip(ps, outs)]
    r = rm_score(texts)  # [B]

    # microbatch logprobs to avoid OOM even on tiny GPU
    B = full.size(0)
    logp_pi_all, T_all, logp_ref_all = [], [], []
    for i in range(B):
        mb = full[i:i+1].to(device)
        logp_pi, T = sum_logprob_generated_safe(policy, mb, inlen)
        with torch.no_grad():
            logp_ref, _ = sum_logprob_generated_safe(pi_ref, mb, inlen)
        logp_pi_all.append(logp_pi); T_all.append(T); logp_ref_all.append(logp_ref)
        del mb
    logp_pi = torch.cat(logp_pi_all, 0)
    T = torch.cat(T_all, 0)
    logp_ref = torch.cat(logp_ref_all, 0)

    kl_tok = (logp_pi.detach() - logp_ref) / T.detach()
    r_total = r - cfg.beta_kl * kl_tok

    adv = (r_total - r_total.mean()).detach()
    loss = -(adv * logp_pi).mean()

    opt_pi.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
    opt_pi.step()

    logs.append({
        "step": int(step),
        "loss": float(loss.item()),
        "rm": float(r.mean().item()),
        "kl_tok": float(kl_tok.mean().item()),
        "r_total": float(r_total.mean().item()),
    })

    if (step+1) % 10 == 0:
        pbar.set_postfix({"rm": f"{logs[-1]['rm']:.3f}", "kl/tok": f"{logs[-1]['kl_tok']:.3f}"})
    clear_cuda()

pi_dir = os.path.join(cfg.out_dir, "policy_rlhf")
os.makedirs(pi_dir, exist_ok=True)
policy.save_pretrained(pi_dir, safe_serialization=True)
tokenizer.save_pretrained(pi_dir)
with open(os.path.join(cfg.out_dir, "logs.json"), "w") as f:
    json.dump(logs, f, indent=2)

print("Saved RLHF policy:", pi_dir)

Tiny RLHF:   0%|          | 0/80 [00:00<?, ?it/s]

Saved RLHF policy: ./tiny_socratic_rlhf/policy_rlhf


In [ ]:
policy.eval(); pi_ref.eval()

def gen(model, prompt):
    return generate_one(model, prompt, temp=0.9)

idxs = np.random.choice(len(prompt_texts), size=6, replace=False)
for k, idx in enumerate(idxs, start=1):
    p = prompt_texts[int(idx)]
    ref = gen(pi_ref, p)
    rl  = gen(policy, p)
    print("="*70)
    print("Topic:", PROMPTS[int(idx)][0])
    print("Misconception:", PROMPTS[int(idx)][1])
    print("REF :", ref)
    print("RLHF:", rl)

Topic: Neural Nets
Misconception: Deeper networks always overfit more.
REF : Rhpress dispatch Motorola Motorolaiken Prob directly TA directly Jr Rh pawn Daniel Brewiken Money subst scalp Rh stairspress?
RLHF: Money conservationoho Daniel possibility ESV directlyatisf heir TA credibilityting004 hauled scalp Habit Jr Habit intermittentditRocket hauled credibility stairs?
Topic: Info Theory
Misconception: Entropy is maximized when probabilities are unequal.
REF : reement circumcised Hancock vendors Participationmediately TA conservationScene vendors Observ Habit Brewreement Prob credibility HabitScene Brew Money Daniel substSher?
RLHF: TA Prob Brew circumciseddit heirdit ESV circumcised ESV reviewing antibiotic autonomypress autonomyhibit TA credibility Brew conservationdit reborn trilogyreementohoScene?
Topic: Linear Algebra
Misconception: If Ax=b has a solution, it must be unique.
REF : intermittent scalp TAJD Motorola reviewing Motorola antibioticpress Jr vendors ESVJD Prob Habit Brewd

In [ ]:
ab_path = os.path.join(cfg.out_dir, "ab_eval.jsonl")

def human_ab(n=10):
    wins = 0; total = 0
    idxs = np.random.choice(len(prompt_texts), size=min(n, len(prompt_texts)), replace=False)
    for k, idx in enumerate(idxs, start=1):
        p = prompt_texts[int(idx)]
        ref = generate_one(pi_ref, p, temp=0.9)
        rl  = generate_one(policy, p, temp=0.9)

        # shuffle order
        if random.random() < 0.5:
            A, B = ref, rl
            Ais, Bis = "REF", "RLHF"
        else:
            A, B = rl, ref
            Ais, Bis = "RLHF", "REF"

        print("="*70)
        print(f"[{k}/{len(idxs)}] Topic:", PROMPTS[int(idx)][0])
        print("Misconception:", PROMPTS[int(idx)][1])
        print("A:", A)
        print("B:", B)

        c=None
        while c not in ("0","1","2"):
            c=input("Which is better? 1=A,2=B,0=skip: ").strip()
        if c in ("1","2"):
            total += 1
            chosen = Ais if c=="1" else Bis
            if chosen=="RLHF":
                wins += 1

        with open(ab_path, "a", encoding="utf-8") as f:
            f.write(json.dumps({
                "prompt_id": int(idx),
                "A": A, "B": B, "choice": int(c),
                "A_is": Ais, "B_is": Bis
            }, ensure_ascii=False) + "\n")

    print("\nRLHF win-rate:", (wins/total if total else None), f"({wins}/{total})")
    print("Saved:", ab_path)

human_ab(10)

[1/10] Topic: Graphs
Misconception: High-degree nodes must be close in shortest path.
A: brutality grandchildren courtyardpublic448 praying rubbing Late448 braveryOutsideived clearer representations Wheels WheelsMost TelevisionMini braveryacious incarcer equatePros Televisionacious Boone factors?
B: vendorsSher hauled Rh stairs intermittentScene Daniel vendors circumcisedRocket TAScene intermittent Habitmediately ONEreement hauled directly Motorola scalp directly scalp?
Which is better? 1=A,2=B,0=skip: 2
[2/10] Topic: Linear Algebra
Misconception: If Ax=b has a solution, it must be unique.
A: Rh Observpressdit directlyScenehibitdit stairs directlymediately autonomy004atisf Observpress directly antibiotic Habit directly antibiotic heiroho Daniel hauledreementRocket scalp scalp Observ?
B: hibit ONE ONE004 confirpress TA conservationSher reviewing conservation Observoho ONE TAScene conservation Money reviewing004 circumcised reborn Habit Observ?
Which is better? 1=A,2=B,0=skip: 0
[3/10] T

In [ ]:
print("Output dir:", cfg.out_dir)
print("prefs:", os.path.join(cfg.out_dir, "prefs.jsonl"))
print("rm head:", os.path.join(cfg.out_dir, "rm", "head.pt"))
print("policy:", os.path.join(cfg.out_dir, "policy_rlhf"))
print("logs:", os.path.join(cfg.out_dir, "logs.json"))
print("ab eval:", os.path.join(cfg.out_dir, "ab_eval.jsonl"))

Output dir: ./tiny_socratic_rlhf
prefs: ./tiny_socratic_rlhf/prefs.jsonl
rm head: ./tiny_socratic_rlhf/rm/head.pt
policy: ./tiny_socratic_rlhf/policy_rlhf
logs: ./tiny_socratic_rlhf/logs.json
ab eval: ./tiny_socratic_rlhf/ab_eval.jsonl
